> <p><small>This notebook is made available subject to the licence and terms set out in <a href="https://creativecommons.org/licenses/by/4.0">https://creativecommons.org/licenses/by/4.0</a>.</small></p>

<img src="https://pub-bba109a9a6ac49e3b428cdca19c34363.r2.dev/LT%20-%20Session%207.jpg">

# 7.6 AI Long Activity 1 - Lab: Compute, Memory and Efficiency (Teacher)

Develop an operational understanding of why training can become slow or infeasible.

60 minutes

## Overview
You will estimate compute and memory requirements, investigate scaling behaviour, and relate simple models to practical optimisation strategies.


## Step 1 – Compute model (forward intuition)

**Predict:** If you double sequence length, what happens to compute?


In [ ]:
# Example computation.
def compute_cost(params, seq_len):
    """Return the estimated compute cost."""
    return params * seq_len


print(
    "100 -> 200:",
    compute_cost(1e6, 100),
    "->",
    compute_cost(1e6, 200),
)

> 🗣️ **Explain:**  
> Linear in sequence length in this toy model (real attention can be worse).



## Step 2 – Scaling sweep

Observe growth over orders of magnitude.


In [ ]:
# Example computation.
for seq in [100, 1_000, 10_000]:
    print(seq, compute_cost(1e6, seq))


**Insight:** Multiplicative scaling → rapid growth.


## Step 3 – Memory = parameters + activations

Separate memory into:
- Parameters (≈ params × bytes)
- Activations (≈ batch_size × sequence_length × num_layers × d_model)

This mirrors the simplified scaling law introduced in the online curriculum.


In [ ]:
# Step 3.
# Memory decomposition example.


def memory_params(params, bytes_per_param=4):
    """Estimate memory used by model parameters."""
    return params * bytes_per_param


def memory_activations(
    batch_size,
    max_length,
    num_layers=12,
    d_model=768,
    bytes_per_param=4,
):
    """Estimate activation memory during forward/backward passes."""
    return (
        batch_size
        * max_length
        * num_layers
        * d_model
        * bytes_per_param
    )


def memory_total(params, batch_size, max_length):
    """Combine parameter and activation memory estimates."""
    return memory_params(params) + memory_activations(batch_size, max_length)


print(memory_total(1e7, 4, 128))

> 🗣️ **Explain:**  
> Activations scale with **batch × sequence** → often dominate during training.


## Step 4 – Batch size impact

> 🔮 **Predict:**  
> What happens to memory when batch increases?


In [ ]:
# Example computation.
for b in [1, 4, 16, 64]:
    print("batch", b, "memory", memory_total(1e7, b, 100))

> 🗣️ **Explain:**  
> Activations term grows with batch → memory pressure.


## Step 5 – Sequence length impact

**Predict:** Which grows faster for memory: params or sequence?


In [ ]:
# Example computation.
for s in [100, 500, 1000]:
    print("seq", s, "memory", memory_total(1e7, 4, s))

> 🗣️ **Explain:**  
> Activations scale with sequence; long contexts are expensive.



## Step 6 – Diagnose bottlenecks

**Signals:**
- GPU util high → compute-bound
- memory near 100% + low util → memory-bound
- GPU idle → data-bound

**Question:** Classify each scenario.



A) util low, memory ~100% → **memory-bound**  
B) util high, slow epochs → **compute-bound**  
C) GPU idle → **data-bound**


## Step 7 – Gradient accumulation

> 🗣️ **Idea:**  
> Keep small batch (fit memory), accumulate steps to emulate larger batch.


In [ ]:
# Example computation.
def effective_batch(batch, steps):
    """Return the effective batch size."""
    return batch * steps


print("effective batch:", effective_batch(4, 8))

> 🗣️ **Explain:**  
> Trades **time for memory** (more steps, same memory).



## Step 8 – Feasibility check (16 GB GPU)

Estimate parameter memory (float32 ≈ 4 bytes).


In [ ]:
# Example computation.
def bytes_to_gb(x):
    """Convert bytes to gibibytes."""
    return x / (1024 ** 3)


for params in [1e7, 1e9, 1e10]:
    print(
        params,
        "params ->",
        round(bytes_to_gb(memory_params(params)), 2),
        "GB",
    )

> 🗣️ **Explain:**  
> 10B params ≈ 40 GB → does not fit on 16 GB (without tricks).


## Step 9 – Decision task

**Constraint:** 16 GB GPU, long sequences needed, latency moderate.

Choose two changes and justify:
- Reduce batch.
- Whorten sequence.
- Smaller model.
- Gradient accumulation.
- Lower precision.



## Final synthesis

Students should conclude:
- Scaling (params, seq, batch) drives cost.
- Activations often dominate memory in training.
- Correct diagnosis → correct fix.
- All solutions are trade-offs.
